# Dynamic Lot-Size Model

This is a classic problem of balancing inventory holding and setup
costs, originally proposed by Wagner & Whitin (1958). In each period
$t \in T = \{1, \dots, |T|\}$, a demand $d_{t}$ must be met by the
production $x_{t}$ or by the inventory $I_{t-1}$ carried from the
previous period. Producing in period $t$ requires a setup ($y_{t} = 1$)
with cost $s_{t}$, and the inventory at the end of the period has a
holding cost $h_{t}$ per unit. The initial inventory is $I_{0} = 0$, and
the total demand $M = \sum_{t \in T} d_{t}$ bounds the production in a
period.

$$
\begin{aligned}
    \text{min} \quad & \sum_{t \in T} (h_{t} I_{t} + s_{t} y_{t}) \\
    \text{s.t.} \quad & I_{t} = I_{t - 1} + x_{t} - d_{t} & \forall ~ t \in T \\
    & x_{t} \leq M y_{t} & \forall ~ t \in T \\
    & x_{t}, I_{t} \geq 0 & \forall ~ t \in T \\
    & y_{t} \in \{0, 1\} & \forall ~ t \in T
\end{aligned}
$$

It is solved here with the naive MILP Branch & Bound of
`bnbprob.milpy`, in which the LP relaxation of each node is solved by
`scipy`.

In [1]:
import numpy as np
import pandas as pd

from bnbprob.milpy import MILP
from bnbpy import BranchAndBound

In [2]:
data = pd.read_csv("../data/lot_size.csv")
print(f"{len(data)} periods")
data.head()

12 periods


,time,setup_cost,inventory_cost,demand
0,1,85,1.0,69
1,2,102,1.0,29
2,3,102,1.0,36
3,4,101,1.0,61
4,5,98,1.0,61


## Write Problem in Matrix Form

`MILP` follows the conventions of `scipy.optimize.linprog`. The vector
of variables is $(x_{1}, \dots, x_{|T|}, I_{1}, \dots, I_{|T|},
y_{1}, \dots, y_{|T|})$, and:

* The inventory balance is written as $x_{t} + I_{t-1} - I_{t} = d_{t}$
  (equality constraints, $I_{0} = 0$).
* The setup constraints are written as $x_{t} - M y_{t} \leq 0$
  (inequality constraints).
* Only the setup variables $y_{t}$ are integer, with bounds $[0, 1]$.

In [3]:
def create_lot_size(
    d: np.ndarray,
    s: np.ndarray,
    h: np.ndarray,
    **kwargs
) -> MILP:
    # Number of periods and big M
    T = len(d)
    M = sum(d)

    # 1. Cost vector: c = [0, ..., 0, h1, ..., hT, s1, ..., sT]
    c = np.concatenate([
        np.zeros(T),  # Production costs (zero in this basic model)
        h,  # Holding costs
        s,  # Setup costs
    ])

    # 2. Equality constraints (inventory balance): A_eq x = b_eq
    A_eq = np.zeros((T, 3 * T))
    b_eq = np.array(d)  # Demand values

    for t in range(T):
        A_eq[t, t] = 1  # Coefficient of x_t
        if t > 0:
            A_eq[t, T + t - 1] = 1  # Coefficient of I_{t-1}
        A_eq[t, T + t] = -1  # Coefficient of I_t

    # 3. Inequality constraints (setup): A_ub x <= b_ub
    A_ub = np.zeros((T, 3 * T))
    b_ub = np.zeros(T)

    for t in range(T):
        A_ub[t, t] = 1  # Coefficient of x_t
        A_ub[t, 2 * T + t] = -M  # Coefficient of y_t

    # Production and inventory are non-negative, setups are binary
    bounds = [(0, None)] * (2 * T) + [(0, 1)] * T

    integrality = np.zeros_like(c)
    integrality[2 * T:] = 1.0

    return MILP(
        c,
        A_eq=A_eq,
        b_eq=b_eq,
        A_ub=A_ub,
        b_ub=b_ub,
        bounds=bounds,
        integrality=integrality,
        **kwargs
    )

## Solve with bnbpy

Nodes are branched on the setup variable with the largest fractional
part (`branching="max"`).

In [4]:
milp = create_lot_size(
    data["demand"].values,
    data["setup_cost"].values,
    data["inventory_cost"].values,
    branching="max",
    seed=42
)

bnb = BranchAndBound(milp)
sol = bnb.solve(maxiter=20000)
print(sol)
print(f"Nodes explored: {bnb.explored}")

Status: OPTIMAL | Cost: 864.0 | LB: 864.0
Nodes explored: 847


In [5]:
T = len(data)
# Adding 0.0 turns -0.0 into 0.0
x = sol.problem.results.x.round(2) + 0.0
plan = pd.DataFrame({
    "time": data["time"],
    "demand": data["demand"],
    "production": x[:T],
    "inventory": x[T:2 * T],
    "setup": x[2 * T:].astype(int),
}).set_index("time")
plan

,demand,production,inventory,setup
time,,,,
1,69,98.0,29.0,1
2,29,0.0,0.0,0
3,36,97.0,61.0,1
4,61,0.0,0.0,0
5,61,121.0,60.0,1
6,26,0.0,34.0,0
7,34,0.0,0.0,0
8,67,112.0,45.0,1
9,45,0.0,0.0,0


## Reference

Wagner, H. M., & Whitin, T. M. (1958). Dynamic version of the economic
lot size model. Management Science, 5(1), 89-96.